<div style="border-left:4px solid #34d399;padding:2px 0 2px 16px;margin:6px 0 18px;"><div style="font:800 27px/1.15 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;letter-spacing:-0.02em;">NL2SQL <span style="font-weight:500;color:#34d399;">Understanding</span></div><div style="font:400 15px/1.55 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#71717a;margin-top:5px;">How an English question becomes a schema, a symbol and an exact value.</div></div>

[Setup](https://www.kaggle.com/code/kirazul/nl2sql-1-setup) &nbsp;|&nbsp; **Understanding** &nbsp;|&nbsp; [Architectures](https://www.kaggle.com/code/kirazul/nl2sql-3-architectures) &nbsp;|&nbsp; [Run All](https://www.kaggle.com/code/kirazul/nl2sql-4-run-all)

## 1. Setup

The code is cloned from GitHub. The database, the index and the two models are
read from [notebook 1](https://www.kaggle.com/code/kirazul/nl2sql-1-setup)'s saved output, where they already are. Nothing
is downloaded or rebuilt here.

Before running: **Add Input > Notebook Output > NL2SQL 1 Setup**, add the secrets
listed below under **Add-ons > Secrets**, and enable Internet.

In [ ]:
%%capture --no-stderr
!pip install -q --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu \
    "llama-cpp-python>=0.3" "gliner2>=1.3" "langgraph>=1.0" "langsmith>=0.10" \
    "fastapi>=0.115" "uvicorn[standard]>=0.34" "pydantic-settings>=2.6" \
    "sqlglot>=25.0" "rapidfuzz>=3.10" "pyyaml>=6.0" "httpx>=0.27" "python-dotenv>=1.0"

In [ ]:
import os, re, sys, json, time, shutil, subprocess
from pathlib import Path

ON_KAGGLE = Path("/kaggle").exists()
WORK      = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
INPUTS    = Path("/kaggle/input")
REPO      = "https://github.com/Kirazul/NL2SQL-demo.git"

SECRETS = {
    "GITHUB_TOKEN":       "clone the code (the repository is private)",
    "GROQ_API_KEY":       "the cloud model that writes the SQL",
    "OPENROUTER_API_KEY": "fallback when Groq rate-limits",
    "LANGSMITH_API_KEY":  "tracing backend",
    "PUBLISH_TOKEN":      "announce this session to the web interface",
}
REQUIRED = ()


WHY = {}          # label -> why it could not be read, when it could not


def secret(label, default=""):
    """One secret, by label. Kaggle grants access per notebook, not per account.

    The reason a lookup failed is kept rather than swallowed: "not attached to
    this notebook" and "the backend refused" both end as an empty string, and
    without the reason the two are indistinguishable from the output.
    """
    if ON_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            value = UserSecretsClient().get_secret(label)
            if value:
                return value
            WHY[label] = "Kaggle returned an empty value"
        except Exception as error:
            WHY[label] = f"{type(error).__name__}: {str(error)[:110]}"
    return os.environ.get(label, default)


def load_secrets(project=None):
    """Read every label into the environment and print what was found.

    An empty secret is removed rather than set blank, so the package falls back to
    its own default instead of an empty string.
    """
    local = {}
    if not ON_KAGGLE and project and (project / ".env").exists():
        for line in (project / ".env").read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                label, _, value = line.partition("=")
                local[label.strip()] = value.strip().strip("\"'")

    for label in SECRETS:
        value = secret(label) or local.get(label, "")
        if value:
            os.environ[label] = value
        else:
            os.environ.pop(label, None)

    for label, purpose in SECRETS.items():
        if os.environ.get(label):
            state = "ok"
        elif label in REQUIRED:
            state = "REQUIRED"
        else:
            state = "-"
        print(f"  {label:<20}{state:<10}{purpose}")

    if WHY:
        print("\n  why a secret could not be read")
        for label, reason in WHY.items():
            print(f"    {label:<20}{reason}")

    absent = [l for l in REQUIRED if not os.environ.get(l)]
    if absent:
        where = "Add-ons > Secrets, in this notebook" if ON_KAGGLE else ".env"
        print(f"\n  Missing: {', '.join(absent)}. Set it in {where} and run this cell again.")
    return not absent


def get_code():
    """Clone the repository into a writable directory and put it on the path.

    Kaggle mounts every input read-only and notebook 1 writes a database next to
    the package, so the code never runs from where it is mounted.
    """
    if (Path.cwd() / "src/hybridsql").exists():
        return Path.cwd()

    target = WORK / "nl2sql"
    if (target / "src/hybridsql").exists():
        return target

    token = secret("GITHUB_TOKEN")
    url = REPO.replace("https://", f"https://{token}@") if token else REPO
    done = subprocess.run(["git", "clone", "--depth", "1", "--quiet", url, str(target)],
                          capture_output=True, text=True)
    if done.returncode:
        detail = done.stderr.replace(token, "***") if token else done.stderr
        raise SystemExit(
            "Could not clone the repository.\n\n"
            "  It is private, so this notebook needs a GITHUB_TOKEN secret:\n"
            "  Add-ons > Secrets > attach GITHUB_TOKEN, then run this cell again.\n\n"
            "  Kaggle grants a secret one notebook at a time. Attaching it in\n"
            "  another notebook does not attach it here.\n\n" + detail
        )
    return target
ARTEFACTS = {
    "database":    ("data/warehouse/eicu.db",                   None),
    "value index": ("data/warehouse/value_index.db",            None),
    "GLiNER2":     ("models/gliner2-base-v1",                   "model.safetensors"),
    "Qwen3-1.7B":  ("models/qwen3-1.7b/Qwen3-1.7B-Q4_K_M.gguf", None),
}
DEPTHS = ("", "*/", "*/*/", "*/*/*/", "*/*/*/*/", "*/*/*/*/*/")


def whole(path, probe=None):
    """Present and finished. A model directory with no weights in it is neither."""
    return (path / probe).exists() if probe else path.exists()


def find_input(relative, probe=None):
    """The first attached input carrying `relative`, at whatever depth it sits."""
    if not INPUTS.exists():
        return None
    for prefix in DEPTHS:
        for hit in sorted(INPUTS.glob(prefix + relative)):
            if whole(hit, probe):
                return hit
    return None


def attached_inputs():
    """The inputs actually attached, named by what they carry rather than by the
    directory level Kaggle happens to mount them under."""
    if not INPUTS.exists():
        return []
    markers = ("src", "data", "models", "nl2sql")
    return [p.relative_to(INPUTS).as_posix()
            for pattern in ("*", "*/*", "*/*/*")
            for p in sorted(INPUTS.glob(pattern))
            if p.is_dir() and any((p / m).exists() for m in markers)]


def locate(project):
    """Every artefact, in the working copy or in an attached input."""
    found, missing = {}, []
    for label, (relative, probe) in ARTEFACTS.items():
        if label == "value index":
            continue                       # always beside the database, see below
        local = project / relative
        path = local if whole(local, probe) else find_input(relative, probe)
        (found.__setitem__(label, path) if path else missing.append(label))

    # The package derives the index path from the database path, so the two must
    # be in the same directory. Looking for it anywhere else would resolve here
    # and fail there.
    if "database" in found:
        index = found["database"].with_name("value_index.db")
        found["value index"] = index if index.exists() else missing.append("value index")
    else:
        missing.append("value index")
    return found, [m for m in missing if m]


def configure(found):
    os.environ["DB_PATH"]             = str(found["database"])
    os.environ["GLINER_MODEL"]        = str(found["GLiNER2"])
    os.environ["LOCAL_LLM_GGUF_PATH"] = str(found["Qwen3-1.7B"])
    os.environ["LOCAL_LLM_THREADS"]   = str(max(2, os.cpu_count() or 4))
    os.environ["LOCAL_LLM_BACKEND"]   = "llamacpp"
    os.environ["PRIVACY_MODE"]        = "demo"
    os.environ["LANGSMITH_PROJECT"]   = "nl2sql"
    os.environ["LANGSMITH_TRACING"]   = "1" if os.environ.get("LANGSMITH_API_KEY") else "0"


def size_mb(path):
    if path.is_dir():
        return sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) / 1e6
    return path.stat().st_size / 1e6 if path.exists() else 0.0


def show(found):
    for label, (relative, _) in ARTEFACTS.items():
        path = found.get(label)
        if path is None:
            print(f"  {label:<14}{'missing':>10}")
            continue
        root = path.parents[len(Path(relative).parts) - 1]
        if WORK in path.parents:
            where = "built here"
        elif INPUTS.exists() and (INPUTS in root.parents or root == INPUTS):
            where = root.relative_to(INPUTS).as_posix()
        else:
            where = str(root)
        print(f"  {label:<14}{size_mb(path):>9.0f} MB   {where}")
print("code")
PROJECT = get_code()
sys.path.insert(0, str(PROJECT / "src"))
os.chdir(PROJECT)
print(f"  {PROJECT}")

print("\nsecrets")
load_secrets(PROJECT)

FOUND, MISSING = locate(PROJECT)
if MISSING:
    raise SystemExit(
        "Notebook 1's output is not attached, and nothing is built in this notebook.\n"
        f"  missing:  {', '.join(MISSING)}\n"
        f"  attached: {attached_inputs() or 'nothing'}\n\n"
        "  Add Input > Notebook Output > NL2SQL 1 Setup\n"
        "  https://www.kaggle.com/code/kirazul/nl2sql-1-setup"
    )

configure(FOUND)
print("\nartefacts")
show(FOUND)

---

## 2. The problem

Someone asks, in English:

> *How many patients over 65 received aspirin?*

Answering it means writing SQL, and writing good SQL over 31 unfamiliar tables is
something large cloud models do well and small local models do badly. But sending
the question to a cloud model sends the thing you are trying to protect, because
the question is about the data: it names a drug, an age, a ward.

This notebook shows the way out. By the end of it the question has become a
schema, a set of symbols and an exact stored value, and **nothing has been sent
anywhere**. What can then be sent is the subject of notebook 3.

Four stages, all local, in this order:

| Stage | Does | Module |
|---|---|---|
| **Extract** | finds the entities in the sentence | `pipeline/understand.py` |
| **Resolve** | turns each one into a real database value | `db/value_index.py` |
| **Mask** | replaces every value with a symbol | `pipeline/anonymize.py` |
| **Verify** | checks every outgoing word before a socket opens | `security/egress_gate.py` |

---

## 3. The data

**eICU-CRD** is 31 tables of intensive-care records: patients, their stays, the
drugs they were given, their lab results, their diagnoses.

Almost every table hangs off one column. `patientunitstayid` identifies a single
patient's stay in a single ICU, and it appears in 28 of the 31 tables. That one
column is what makes a question like *which patients on aspirin had a high
creatinine* answerable: `medication` and `lab` both carry it, so they join.

In [ ]:
from hybridsql.db import schema as sch

for key, value in sch.summary().items():
    print(f"  {key:<16}{value:,}")

tables = sch.read_schema()
print("\n  largest tables")
for name in sorted(tables, key=lambda t: -tables[t].row_count)[:6]:
    print(f"    {name:<20}{tables[name].row_count:>10,} rows   {len(tables[name].columns):>3} columns")

joined = sum(1 for t in tables.values() if "patientunitstayid" in t.columns)
print(f"\n  {joined} of {len(tables)} tables carry patientunitstayid")

### The schema, as the cloud model receives it

This is the entire disclosure of the Hybrid architecture: table names, column
names, types, row counts and the foreign keys. No value from any row.

In [ ]:
print(sch.ddl({"patient", "medication"}, with_row_counts=True)[:850])

---

## 4. Stage 1, extract the entities

GLiNER2 reads the sentence and returns the spans that carry meaning, each with a
confidence. It runs in this process, on the CPU, and it is zero-shot: the types
below are described in English at call time, not trained in.

The model does not know what a `drugname` column is. It knows the sentence
contains a drug, and where.

In [ ]:
from hybridsql.pipeline.understand import understand

u = understand("How many patients over 65 received aspirin?")

print(f"  extractor  {u.active_extractor}")
print(f"  tables     {', '.join(sorted(u.tables))}\n")
print(f"  {'span':<14}{'type':<12}{'confidence':>11}")
for r in u.resolutions:
    print(f"  {r.mention:<14}{r.kind:<12}{r.score:>11.2f}")

---

## 5. Stage 2, resolve them against the database

*aspirin* is not a value. `ASPIRIN EC 81 MG PO TBEC` is. Something has to bridge
them without asking a cloud model what the database contains, and that something
is an index built once by notebook 1.

### Why not simply index everything

Because the cost would then grow with the number of rows, and a privacy design
that stops working on a large database is not a design. So every text column is
measured once and sorted into one of three tiers.

| Tier | The column looks like | What is stored |
|---|---|---|
| **A** | a bounded vocabulary: 6 drug note types, 12 wards | every distinct value |
| **B** | high cardinality: thousands of distinct strings | nothing, searched on demand |
| **C** | free text, identifiers, constants | nothing, never searched |

The stored size is therefore `columns x vocabulary limit`, and adding ten million
rows to a table adds nothing to the index as long as the set of distinct values
does not grow.

In [ ]:
from hybridsql.db import value_index

s = value_index.stats()
print(f"  columns examined   {s['tiers']['A'] + s['tiers']['B'] + s['tiers']['C']}")
print(f"  tier A, indexed    {s['tiers']['A']}")
print(f"  tier B, on demand  {s['tiers']['B']}")
print(f"  tier C, excluded   {s['tiers']['C']}")
print(f"  values stored      {s['values_indexed']:,}")
print(f"  index size         {s['size_mb']} MB")

### The decision, column by column

The classification is written down when the index is built, so it can be read
back and argued with. These are real columns from the database.

In [ ]:
report = json.loads(Path(os.environ["DB_PATH"]).with_name("column_classification.json").read_text())
columns = report["columns"]

print(f"  {'column':<42}{'tier':<6}{'distinct':>9}   reason")
for tier in ("A", "B", "C"):
    for c in [c for c in columns if c["tier"] == tier][:3]:
        print(f"  {c['ref'][:40]:<42}{c['tier']:<6}{c['distinct']:>9}   {c['reason']}")

### Resolving one word

`medication.drugname` is a tier A column with a few thousand distinct spellings.
Asking the index for *aspirin* returns the real values, the column each came
from, and a score.

In [ ]:
for hit in value_index.search("aspirin", limit=5):
    print(f"  {hit.score:.2f}  {hit.ref:<28}{hit.value}")

Two things came back, and the second matters as much as the first: the exact
value **and the column it belongs to**. Knowing that `:v1` is a value of
`medication.drugname` is what later lets the opaque architecture describe the
query without naming the drug or the column.

Here is the full resolution for our question.

In [ ]:
print(f"  {'span':<14}{'column':<28}resolved to")
for r in u.resolutions:
    print(f"  {r.mention:<14}{str(r.column or '-'):<28}{r.value or '-'}")

---

## 6. Stage 3, mask

Every resolved value is replaced by a symbol. `ASPIRIN EC 81 MG PO TBEC` becomes
`:v1`, `65` becomes `:v2`, and the mapping between them stays in this process.

The symbols are renumbered on every request, so `:v1` in one question and `:v1`
in the next are unrelated. Nobody watching the outgoing traffic can follow a
value across two questions.

In [ ]:
from hybridsql.pipeline.anonymize import anonymize

a = anonymize(u)

print(f"  asked   {u.question}")
print(f"  sent    {a.masked_question}\n")
print("  kept here")
for symbol, value in a.mapping.items():
    print(f"    {symbol:<6}{value!r:<32}{a.columns.get(symbol, '')}")

---

## 7. Stage 4, verify

The masking is the design. The gate is what makes it checkable.

One rule holds the system up: **exactly one module may open a socket, and every
piece of text it sends is verified first**. The outgoing prompt is not checked as
one blob. It is split by where each part came from, and each part is checked by
the rule that can actually prove that kind of text safe.

| Origin | Proven safe by |
|---|---|
| `authored` | matching the fingerprint of a literal in the source |
| `template` | matching the fingerprint of the wording |
| `schema` | regenerating it from the database and comparing |
| `glossary` | membership in the declared notes |
| `question` | word by word, the only untrusted part |

In [ ]:
from hybridsql.pipeline import generate as gen
from hybridsql.security import egress_gate

for segment in gen.build_segments(u, a):
    verdict = egress_gate.check_segment(segment, "notebook")
    mark = "pass " if verdict.allowed else "BLOCK"
    print(f"  [{mark}] {segment.origin:<9}{verdict.verified_by:<26}"
          f"{' '.join(segment.text.split())[:38]}...")

### A real value, submitted on purpose

The gate is only worth something if it refuses. This sends a genuine drug name
through it. The exception is the correct result, and no socket opens.

In [ ]:
from hybridsql.security.egress_gate import LeakBlocked, Segment

try:
    egress_gate.require_segments(
        [Segment("How many patients received AMOXICILLIN 500 MG PO CAPS?", "question")],
        "check")
    print("  ALLOWED. This would be a failure.")
except LeakBlocked as blocked:
    print(f"  refused: {blocked}")

---

## 8. What would leave

The question arrived in English and named a drug. What is now ready to send names
no drug, no patient and no row.

In [ ]:
print(f"  the question    {u.question}")
print(f"  what is sent    {a.masked_question}")
print(f"  values out      0")
print(f"  kept here       {len(a.mapping)} value(s), {len(u.tables)} table name(s)")

---

Four architectures do different things with this. Three send it, one does not, and the difference is measurable.

**Next:** [3. Architectures](https://www.kaggle.com/code/kirazul/nl2sql-3-architectures)